# Simple Logic Gates with a Perceptron

This notebook demonstrates how a **single-layer perceptron** can learn linearly separable logic gates (AND) and explores why it fundamentally **fails on XOR** — a non-linearly separable problem.

The progression below builds intuition for why deeper, multi-layer networks are necessary.

* Starting with creating a single Perceptron Class

In [48]:
import numpy as np

class MyPerceptron:
    def __init__(self, learning_rate=0.1, epochs=10):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None

    def activation_function(self, x):
        # Step Function
        return 1 if x >= 0 else 0

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Zero initialization of weights and bias
        self.weights = np.zeros(n_features)
        self.bias = 0

        for epoch in range(self.epochs):
            for idx, x_i in enumerate(X):
                # 1.Net input calculation z = w*x + b
                linear_output = np.dot(x_i, self.weights) + self.bias
                # 2. Activation function application y_predicted = f(z)
                y_predicted = self.activation_function(linear_output)
                
                # 3. Update weights and bias: w = w + lr * (y - y_pred) * x
                update = self.lr * (y[idx] - y_predicted)
                self.weights += update * x_i
                self.bias += update

    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        y_predicted = [self.activation_function(i) for i in linear_output]
        return np.array(y_predicted)

## Testing the Perceptron on the AND Gate

The AND gate output is `1` only when both inputs are `1`. This is **linearly separable** — a single line divides the input space correctly. 

Train the perceptron and verify it learns the correct weights.

In [49]:
# Training data for AND gate
X1 = np.array([[0,0], [0,1], [1,0], [1,1]])
y1 = np.array([0, 0, 0, 1])

# Create and train the perceptron
p1 = MyPerceptron(learning_rate=0.1, epochs=10)
p1.fit(X1, y1)

print(f"Weights: {p1.weights}")
print(f"Bias: {p1.bias}")
print(f"AND Actual: {y1}")
print(f"Predictions: {p1.predict(X1)}")

Weights: [0.2 0.1]
Bias: -0.20000000000000004
AND Actual: [0 0 0 1]
Predictions: [0 0 0 1]


## The XOR Problem: A Challenge for Single-Layer Perceptrons

XOR is **not** linearly separable — no single straight line can correctly separate all four input combinations. Let's see what happens when we apply our perceptron directly to XOR.

> **Expected outcome:** the model will not converge to 100 % accuracy. This motivates the need for hidden layers.

In [50]:
# XOR Dataset
X2 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y2 = np.array([0, 1, 1, 0])

# create Single Layer Perceptron
# 
model = MyPerceptron(learning_rate=0.1, epochs=10)
model.fit(X2, y2)

print(f"Weights: {model.weights}")
print(f"Bias: {model.bias}")
print(f"XOR Actual: {y2}")
print(f"Predictions: {model.predict(X2)}")



Weights: [-0.1  0. ]
Bias: 0.0
XOR Actual: [0 1 1 0]
Predictions: [1 1 0 0]


## Confirming the Limit with Scikit-learn's Perceptron

The failure is not an implementation bug — it is a mathematical property of single-layer linear classifiers. sklearn's `Perceptron` produces the same result: it can never reach more than 50 % accuracy on XOR because no hyperplane can separate the classes.

In [51]:
import numpy as np
from sklearn.linear_model import Perceptron

# XOR Dataset
X3 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y3 = np.array([0, 1, 1, 0])

# create Single Layer Perceptron
model = Perceptron(max_iter=1000, tol=1e-3)
model.fit(X3, y3)

# Results
print("XOR Actual: ", y3)
print("Perceptron XOR Prediction:", model.predict(X3))
print("Success Rate: %", model.score(X3, y3) * 100)

XOR Actual:  [0 1 1 0]
Perceptron XOR Prediction: [0 0 0 0]
Success Rate: % 50.0


## Solution: Multi-Layer Perceptron with Backpropagation

To solve XOR we need **at least one hidden layer**. A single hidden layer with two neurons is enough to carve out the correct non-linear boundary.

This implementation uses:
- **Architecture**: 2 inputs → 2 hidden neurons → 1 output neuron
- **Activation**: Sigmoid at every layer (smooth, differentiable)
- **Learning rule**: Backpropagation — chain-rule gradients flow backwards to update every weight

After enough epochs the network learns to map the four XOR inputs to the correct outputs.

In [60]:
import numpy as np

# Activation Function and derivative
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

# XOR Data
X4 = np.array([[0,0], [0,1], [1,0], [1,1]])
y4 = np.array([[0], [1], [1], [0]])

# Weight init
# Hidden Layer -> Input
w_hidden = np.random.uniform(size=(2, 2))
# Hidden Layer -> Output
w_output = np.random.uniform(size=(2, 1))

epochs = 10000
lr = 0.5 # Learn Rate

for epoch in range(epochs):
    # Forward Propagation
    hidden_layer_input = np.dot(X4, w_hidden)
    hidden_layer_output = sigmoid(hidden_layer_input)
    
    final_input = np.dot(hidden_layer_output, w_output)
    predicted_output = sigmoid(final_input)
    
    # Error Calculation
    error = y4 - predicted_output
    
    # Backpropagation
    d_predicted_output = error * sigmoid_derivative(predicted_output)
    
    error_hidden_layer = d_predicted_output.dot(w_output.T)
    d_hidden_layer = error_hidden_layer * sigmoid_derivative(hidden_layer_output)
    
    # Wight update
    w_output += hidden_layer_output.T.dot(d_predicted_output) * lr
    w_hidden += X4.T.dot(d_hidden_layer) * lr

print("*Actual XOR Output:", y4.flatten())
print("XOR Predictions:", predicted_output.flatten())
rounded_predictions = np.round(predicted_output).flatten()
print("Rounded XOR Predictions:", rounded_predictions)
rounded_predictions = (predicted_output > 0.5).astype(int).flatten()
print("*Rounded XOR Predictions:", rounded_predictions)

*Actual XOR Output: [0 1 1 0]
XOR Predictions: [0.05356365 0.89932424 0.89932356 0.13372174]
Rounded XOR Predictions: [0. 1. 1. 0.]
*Rounded XOR Predictions: [0 1 1 0]
